# FastReID: crop -> embedding (corrected)

This notebook loads a FastReID model and turns a person crop into an **embedding**
(a 2048-d vector). Same person -> similar vectors; different people -> different
vectors. Similarity is measured with **cosine** (a dot product after we scale
each vector to length 1).

### The three bugs this notebook fixes vs the first draft
1. **Weights were never loaded.** `build_model(cfg)` builds only the *architecture*
   (its own docstring says "it does not load any weights"). You must then call
   `Checkpointer(model).load(path)`. Without it, weights are random and every
   embedding is noise.
2. **Double normalisation.** FastReID normalises *inside* the model and expects
   raw **0-255 RGB** (it subtracts `pixel_mean ~= [123.7, 116.3, 103.5]`). The
   draft divided by 255 first -> the model then subtracted ~123 from numbers <=1.
   Fix: keep pixels in 0-255; let the model normalise.
3. **Hard-coded input size.** The crop must be resized to the size the model was
   trained at, which lives in `cfg.INPUT.SIZE_TEST`. Fix: read it from the config.

> **Weights status:** we have **not** trained a ReID model yet. Below we run the
> **ImageNet-pretrained backbone** as a baseline so the code path is provable
> today. It will *not* separate identities well (that's the whole point of
> training). The real weights come from training on **RandPerson** (Apache-2.0,
> synthetic, commercial-safe); swap them in at the marked line.

In [26]:
import sys, os
from pathlib import Path

# FastReID is vendored in ../fast-reid (a clone), not pip-installed -> add to path.
PROJECT_ROOT = Path.cwd().parent
FASTREID_PATH = PROJECT_ROOT / "fast-reid"
if str(FASTREID_PATH) not in sys.path:
    sys.path.insert(0, str(FASTREID_PATH))

import torch
import numpy as np
import cv2

print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"   # SAME code runs on both
print("Using device:", DEVICE)

Torch: 2.12.1+cu130 | CUDA available: False
Using device: cpu


## 1. Build the config

The config defines both the **architecture** and the **input size** the crops
must match. We use `Base-bagtricks.yml` because it already sets
`BACKBONE.PRETRAIN: True` (ImageNet) and `INPUT.SIZE_TEST: [256, 128]`.

In [27]:
from fastreid.config import get_cfg

CONFIG_FILE = str(FASTREID_PATH / "configs" / "Base-bagtricks.yml")

# ---- point this at trained ReID weights when you have them ----
# WEIGHTS = str(PROJECT_ROOT / "models" / "fastreid" / "model_final.pth")  # after RandPerson training
WEIGHTS = None   # None -> ImageNet-backbone baseline (proves the code, not a real ReID model)

cfg = get_cfg()
cfg.merge_from_file(CONFIG_FILE)
cfg.MODEL.DEVICE = DEVICE

# The classifier head needs a class count to build. At inference we only read the
# embedding (not the classifier), so any positive number is fine.
cfg.MODEL.HEADS.NUM_CLASSES = max(1, cfg.MODEL.HEADS.NUM_CLASSES)

if WEIGHTS:
    cfg.MODEL.WEIGHTS = WEIGHTS
else:
    cfg.MODEL.BACKBONE.PRETRAIN = True   # start backbone from ImageNet

cfg.freeze()

# SIZE_TEST is [Height, Width]; cv2.resize wants (Width, Height) -- easy to swap.
HEIGHT, WIDTH = cfg.INPUT.SIZE_TEST
print("Input size (HxW):", HEIGHT, WIDTH, "| weights:", WEIGHTS or "ImageNet backbone only")

Input size (HxW): 256 128 | weights: ImageNet backbone only


## 2. Build the model AND load the weights (the fix)

`build_model` returns the architecture with **random** weights. `Checkpointer.load`
is the line the first draft was missing.

In [28]:
from fastreid.modeling import build_model
from fastreid.utils.checkpoint import Checkpointer

model = build_model(cfg)                 # architecture only (random weights)

if WEIGHTS:
    Checkpointer(model).load(WEIGHTS)    # <-- FIX #1: actually load trained weights
    print("Loaded ReID weights from:", WEIGHTS)
else:
    print("No ReID weights -> using ImageNet backbone baseline.")

model.to(DEVICE).eval()                  # eval() = use BatchNorm running stats
print("Model ready.")

No ReID weights -> using ImageNet backbone baseline.
Model ready.


## 3. Preprocess + extract one embedding

Four steps, and each must match how the model was trained:
1. resize to `cfg.INPUT.SIZE_TEST`
2. BGR -> RGB (OpenCV loads BGR; model trained on RGB)
3. HWC->CHW float, **keep 0-255** (FIX #2: no `/255`)
4. forward, then L2-normalise (so cosine == dot product)

In [29]:
@torch.no_grad()
def extract(bgr_crop):
    img = cv2.resize(bgr_crop, (WIDTH, HEIGHT))        # (1) trained size, cv2 wants (W,H)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)         # (2) BGR -> RGB
    t = torch.from_numpy(img).float()                  # (3) keep 0-255, do NOT /255
    t = t.permute(2, 0, 1).unsqueeze(0).to(DEVICE)     #     HWC -> CHW, add batch dim
    feat = model(t)                                     # (4) forward -> [1, 2048]
    feat = torch.nn.functional.normalize(feat, dim=1)  #     L2-normalise
    return feat[0].cpu().numpy()

import glob
sample = sorted(glob.glob(str(PROJECT_ROOT / "crops" / "**" / "*.jpg"), recursive=True))
assert sample, "No crops found -- run main.py first."
v = extract(cv2.imread(sample[0]))
print("crop:", sample[0])
print("embedding shape:", v.shape, "| L2 norm:", round(float(np.linalg.norm(v)), 4), "(should be 1.0)")

crop: /home/seif/Projects/Ema_PersonId/crops/cam_219/id_0002/f000000.jpg
embedding shape: (2048,) | L2 norm: 1.0 (should be 1.0)


## 4. The gallery: embedding -> global id (with EMA)

For each embedding: compare (cosine) to every known identity; if the best match
clears `threshold`, reuse that id and nudge its prototype toward the new vector
(EMA smoothing); otherwise mint a new id. In production this linear scan becomes
**Qdrant** ANN search + time/camera-topology rules -- same logic.

In [30]:
class ReIDEngine:
    def __init__(self, extractor, threshold=0.5, alpha=0.2):
        self.extractor = extractor
        self.threshold = threshold        # min cosine to call it "same person"
        self.alpha = alpha                # EMA weight for prototype updates
        self.gallery = {}                 # global_id -> prototype (unit vector)
        self.next_id = 1

    def identify(self, crop):
        feat = self.extractor(crop)       # already L2-normalised
        best_id, best_score = None, -1.0
        for gid, proto in self.gallery.items():
            score = float(np.dot(feat, proto))   # cosine (both unit length)
            if score > best_score:
                best_id, best_score = gid, score

        if best_id is not None and best_score >= self.threshold:
            updated = (1 - self.alpha) * self.gallery[best_id] + self.alpha * feat
            self.gallery[best_id] = updated / np.linalg.norm(updated)   # keep unit length
            return best_id, best_score

        gid = self.next_id
        self.next_id += 1
        self.gallery[gid] = feat.copy()
        return gid, best_score

print("ReIDEngine defined.")

ReIDEngine defined.


## 5. Sanity test: do same-person crops score higher than different-person?

The one test that catches the random-weights bug. With a *trained* model the GAP
should be large. With the ImageNet baseline it's small (and different-person max
can exceed same-person min) -- which is exactly why we need to train on RandPerson.

In [31]:
import itertools

# group crops by their id_XXXX folder (label includes the camera name)
people = {}
for id_dir in sorted(glob.glob(str(PROJECT_ROOT / "crops" / "*" / "id_*"))):
    imgs = sorted(glob.glob(os.path.join(id_dir, "*.jpg")))[:4]
    if len(imgs) >= 2:
        label = os.path.join(os.path.basename(os.path.dirname(id_dir)), os.path.basename(id_dir))
        people[label] = [extract(cv2.imread(p)) for p in imgs]

same = [float(np.dot(a, b)) for vs in people.values() for a, b in itertools.combinations(vs, 2)]
firsts = {l: vs[0] for l, vs in people.items()}
diff = [float(np.dot(a, b)) for (_, a), (_, b) in itertools.combinations(firsts.items(), 2)]

print(f"SAME person cosine: mean={np.mean(same):.3f}  min={np.min(same):.3f}")
print(f"DIFF person cosine: mean={np.mean(diff):.3f}  max={np.max(diff):.3f}")
print(f"GAP (same-diff)   : {np.mean(same)-np.mean(diff):.3f}  "
      f"({'GOOD' if np.mean(same)-np.mean(diff) > 0.1 else 'WEAK -> train on RandPerson'})")

SAME person cosine: mean=0.917  min=0.800
DIFF person cosine: mean=0.810  max=1.000
GAP (same-diff)   : 0.107  (GOOD)


In [32]:
# gallery in action on the first crop of each person
engine = ReIDEngine(extract, threshold=0.5)
for label in firsts:
    first_img = sorted(glob.glob(str(PROJECT_ROOT / "crops" / label / "*.jpg")))[0]
    gid, score = engine.identify(cv2.imread(first_img))
    print(f"{label:20s} -> global id {gid}  (best score {score:.3f})")

cam_219/id_0002      -> global id 1  (best score -1.000)
cam_219/id_0003      -> global id 1  (best score 0.779)
cam_219/id_0004      -> global id 1  (best score 0.991)
cam_219/id_0005      -> global id 1  (best score 0.840)
cam_219/id_0006      -> global id 1  (best score 0.830)
cam_219/id_0007      -> global id 1  (best score 0.876)
cam_219/id_0008      -> global id 1  (best score 0.838)
cam_219/id_0012      -> global id 1  (best score 0.916)
cam_219/id_0015      -> global id 1  (best score 0.853)
cam_219/id_0017      -> global id 1  (best score 0.913)
cam_224/id_0001      -> global id 1  (best score 0.851)
cam_224/id_0002      -> global id 1  (best score 0.897)
cam_224/id_0006      -> global id 1  (best score 0.908)
cam_224/id_0007      -> global id 1  (best score 0.910)
cam_224/id_0010      -> global id 1  (best score 0.942)
cam_224/id_0011      -> global id 1  (best score 0.896)
